# 프롬프트 엔지니어링(Prompt Engineering)

프롬프트 엔지니어링은 단순히 질문을 던지는 것을 넘어, 모델의 작동 원리와 '인컨텍스트 러닝(In-context Learning)' 능력을 활용해 모델의 출력을 제어하는 프로세스이다. 이는 모델의 파라미터(가중치)를 직접 수정하지 않고도 모델의 성능을 특정 태스크에 맞게 조정하는 방법론이다.

**프롬프트의 핵심 구성 요소:**

효과적인 프롬프트는 일반적으로 다음의 4가지 요소를 포함한다.

* **지시문 (Instruction):** 모델이 수행해야 할 구체적인 작업(예: 요약하라, 분류하라, 번역하라 등).
* **문맥 (Context):** 모델이 작업을 더 잘 수행하도록 돕는 배경 정보나 제약 조건.
* **입력 데이터 (Input Data):** 처리가 필요한 실제 데이터.
* **출력 지시자 (Output Indicator):** 결과물의 형식이나 스타일 지정(예: 표로 정리하라, JSON 포맷으로 출력하라 등).

**프롬프트 엔지니어링의 중요성:**

* **성능 최적화:** 같은 모델이라도 프롬프트에 따라 성능 차이가 극심하다. 잘 설계된 프롬프트는 더 작은 모델로도 큰 모델 수준의 결과를 낼 수 있게 한다.
* **비용 효율성:** 불필요한 토큰 사용을 줄이고, 파인튜닝(Fine-tuning)에 비해 적은 비용으로 도메인 특화 작업을 수행할 수 있다.
* **한계 극복:** 모델의 환각 현상을 줄이고 최신 정보를 반영(RAG와 결합 시)하도록 유도할 수 있다.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os
import json

load_dotenv()
# OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
client = OpenAI()

In [ ]:
response = client.chat.completions.create(
    model='gpt-5.6-luna',
    messages=[
        {
            'role': 'system',
            'content': [
                {
                    'type': 'text',
                    'text': "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n- 원래제목: [송고한 기사제목]\n- 교정제목: [교정한 기사제목]\n- 교정 이유:\n  1. [교정한 부분과 이유]\n  2. [교정한 부분과 이유]\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\"\n- 교정 이유:\n   1. '접종율'은 '접종률'이 맞는 표기입니다.\n   2. '높히기'는 '높이기'로, 맞춤법 오류입니다.\n   3. '대안마련'은 붙여쓰지 않고 '대안 마련'으로 띄어 써야 맞습니다.\n   4. 간결한 어미수정\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\"\n- 교정 이유:\n  - 간결한 어미 수정\n"
                }
            ]
        },
        {
            'role': 'user',
            'content': [
                {
                    'type': 'text',
                    'text': "## Input Data\n입력: 피자설기 유행? 언제까지 갈까?"
                }
            ]
        }
    ],
    response_format={'type':'text'},
    temperature=1,
    max_completion_tokens=2048,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    store=False
)

response

ChatCompletion(id='chatcmpl-EERLZwzegqUa3GU01g04PU6qZm5z0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='- 원래제목: 피자설기는 어떤 맛이야? 너도 먹어봤어?\n- 교정제목: 피자설기, 어떤 맛일까? 먹어 본 사람들의 반응은?\n- 교정 이유:\n  1. 구어체인 ‘어떤 맛이야’를 ‘어떤 맛일까’로 바꿔 기사 제목에 맞는 자연스럽고 중립적인 표현으로 수정했습니다.\n  2. 독자를 직접 지칭하는 ‘너도’는 기사 제목에 적절하지 않아 ‘먹어 본 사람들의 반응은?’으로 바꿨습니다.\n  3. ‘먹어봤어’는 보조 용언 띄어쓰기 원칙에 따라 ‘먹어 본’으로 수정했습니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1787110421, model='gpt-5.6-luna', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=416, prompt_tokens=640, total_tokens=1056, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=241, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=

In [6]:
print(response.choices[0].message.content)

- 원래제목: 피자설기는 어떤 맛이야? 너도 먹어봤어?
- 교정제목: 피자설기, 어떤 맛일까? 먹어 본 사람들의 반응은?
- 교정 이유:
  1. 구어체인 ‘어떤 맛이야’를 ‘어떤 맛일까’로 바꿔 기사 제목에 맞는 자연스럽고 중립적인 표현으로 수정했습니다.
  2. 독자를 직접 지칭하는 ‘너도’는 기사 제목에 적절하지 않아 ‘먹어 본 사람들의 반응은?’으로 바꿨습니다.
  3. ‘먹어봤어’는 보조 용언 띄어쓰기 원칙에 따라 ‘먹어 본’으로 수정했습니다.


In [16]:
def correct_headline(headline, /, *, model='gpt-5.6-luna', temperature=1, top_p=1, max_completion_tokens=2048):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': [
                    {
                        'type': 'text',
                        'text': "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n- 원래제목: [송고한 기사제목]\n- 교정제목: [교정한 기사제목]\n- 교정 이유:\n  1. [교정한 부분과 이유]\n  2. [교정한 부분과 이유]\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\"\n- 교정 이유:\n   1. '접종율'은 '접종률'이 맞는 표기입니다.\n   2. '높히기'는 '높이기'로, 맞춤법 오류입니다.\n   3. '대안마련'은 붙여쓰지 않고 '대안 마련'으로 띄어 써야 맞습니다.\n   4. 간결한 어미수정\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\"\n- 교정 이유:\n  - 간결한 어미 수정\n"
                    }
                ]
            },
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'text',
                        'text': f"## Input Data\n입력: {headline}"
                    }
                ]
            }
        ],
        response_format={'type':'text'},
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0,
        store=False
    )

    return response.choices[0].message.content

In [8]:
print(correct_headline('피자 설기 유행, 언제까지 지속될까'))

- 원래제목: 피자 설기 유행, 언제까지 지속될까
- 교정제목: 피자 설기 열풍, 언제까지 이어질까
- 교정 이유:
  1. ‘유행’보다 ‘열풍’이 독자의 관심을 끌면서도 간결하고 임팩트 있는 표현입니다.
  2. ‘지속될까’는 다소 딱딱하므로 ‘이어질까’로 바꿔 자연스러운 제목으로 다듬었습니다.
  3. 맞춤법과 띄어쓰기에는 특별한 오류가 없습니다.


In [9]:
print(correct_headline('피자 설기 유행, 언제까지 지속될까', model='gpt-5.6-sol'))

- 원래제목: "피자 설기 유행, 언제까지 지속될까"
- 교정제목: "‘피자 설기’ 유행, 언제까지 이어질까"
- 교정 이유:
  1. ‘피자 설기’가 유행 대상을 가리키는 표현임을 명확히 하도록 작은따옴표를 사용했습니다.
  2. ‘유행이 지속되다’보다 자연스럽고 간결한 ‘유행이 이어지다’로 표현을 다듬었습니다.


In [11]:
headlines = [
    '피자 설기 너무 맛있다',
    '성수동 팝업스토어 사람 너무 많아 미어터진다',
    '가을 야구 가는 팀은?'
]

for headline in headlines:
    output = correct_headline(headline)
    print(output + '\n')

- 원래제목: 피자 설기 너무 맛있다
- 교정제목: 피자 설기, “너무 맛있다”

- 교정 이유:
  1. ‘피자 설기’와 평가 표현을 쉼표로 구분해 제목의 가독성을 높였습니다.
  2. ‘너무 맛있다’는 구어체 표현이지만, 인용 부호를 사용해 생생한 반응을 살렸습니다.

- 원래제목: [성수동 팝업스토어 사람 너무 많아 미어터진다]
- 교정제목: [성수동 팝업스토어, 방문객 몰려 북새통]

- 교정 이유:
  1. ‘사람 너무 많아’는 구어체 표현이므로 ‘방문객 몰려’로 다듬어 기사 제목에 맞는 간결한 표현으로 수정했습니다.
  2. ‘미어터진다’는 다소 과장되고 감정적인 표현이므로 ‘북새통’으로 바꿔 혼잡한 상황을 자연스럽고 중립적으로 전달했습니다.
  3. 주제와 상황을 구분하기 위해 ‘성수동 팝업스토어’ 뒤에 쉼표를 넣었습니다.

- 원래제목: 가을 야구 가는 팀은?
- 교정제목: 가을야구에 진출할 팀은?

- 교정 이유:
  1. ‘가을 야구’는 야구계에서 포스트시즌을 뜻하는 관용적 표현이므로 ‘가을야구’로 붙여 쓰는 것이 자연스럽습니다.
  2. ‘가는 팀’보다 ‘진출할 팀’이 포스트시즌 진출 여부를 묻는 기사 제목에 적합하고 의미가 명확합니다.
  3. ‘가을야구에’로 조사를 보완해 문법적으로 자연스럽게 다듬었습니다.



In [23]:
def correct_headline_json(headline, /, *, model='gpt-5.6-luna', temperature=1, top_p=1, max_completion_tokens=2048):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': [
                    {
                        'type': 'text',
                        'text': "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n**반드시 json 객체 형식을 준수하세요.**\n\n{{\n  \"original_headline\": <송고한 기사제목>,\n  \"corrected_headline\": <교정한 기사제목>,\n  \"reasons\": [\n     <교정한 부분과 이유>,\n     <교정한 부분과 이유>,\n  ] \n}}\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n{{\n  \"original_headline\": \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\",\n  \"corrected_headline\": \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\",\n  \"reasons\": [\n    \"'접종율'은 표준어가 아니며 '접종률'이 올바른 표기이다\",\n    \"'높히기'는 맞춤법 오류로 '높이기'로 수정해야 한다\",\n    \"'대안마련'은 띄어 써야 하므로 '대안 마련'으로 수정하였다\",\n    \"기사 제목에 맞게 어미를 간결하게 다듬었다\"\n  ]\n}}\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n{{\n  \"original_headline\": \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\",\n  \"corrected_headline\": \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\",\n  \"reasons\": [\n    \"기사 제목의 문체에 맞게 불필요한 어미를 제거해 간결하게 수정하였다\"\n  ]\n}}\n"
                    }
                ]
            },
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'text',
                        'text': f"## Input Data\n입력: {headline}"
                    }
                ]
            }
        ],
        response_format={'type':'json_object'},
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0,
        store=False
    )

    return json.loads(response.choices[0].message.content)

In [24]:
json_responce = correct_headline_json('피자 설기 유행, 언제까지 지속될까')

In [26]:
json_responce

{'original_headline': '피자 설기 유행, 언제까지 지속될까',
 'corrected_headline': '피자 설기 유행, 언제까지 이어질까',
 'reasons': ["'피자 설기'의 띄어쓰기와 표기는 적절하다",
  "'유행'과 '지속되다'는 의미가 다소 중복될 수 있어, 제목 흐름에 맞게 '이어질까'로 다듬어 간결하게 표현했다"]}

In [28]:
def chef_json(user_input, /, *, model='gpt-5.6-luna', temperature=1, top_p=1, max_completion_tokens=2048):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': [
                    {
                        'type': 'text',
                        'text': "# Instruction\n사용자가 입력한 냉장고 내 재료 목록만을 사용하여 만들 수 있는 음식 2가지를 추천하세요.  \n반드시 입력된 재료만 활용하며, 기본양념(간장, 소금, 설탕, 설탕, 식초, 후추 등)은 언제든 사용할 수 있다고 가정하세요.  \n입력된 목록에 없는 재료(양념 제외)는 절대 사용하지 말고, 추가 재료 없이 조리 가능한 음식만 선정하십시오.  \n음식 종류는 반드시 서로 비슷하지 않은 2가지여야 하며, 각 음식에 대한 자세한 요리 레시피(조리 순서)를 단계별로 포함하세요.\n\n- 먼저, 입력 재료로 만들 수 있는 음식 종류를 논리적으로 검토한 뒤, 각 음식이 왜 가능한지 간단히 설명해 주세요.\n- reasoning(논리 및 설명)과 conclusion(최종 추천 결과)은 반드시 JSON의 개별 필드로 구분하여 제공하십시오.\n- reasoning이 반드시 먼저, conclusion이 반드시 마지막에 위치해야 합니다.\n- 결론(conclusion)에는 각 음식명과 단계별 레시피를 포함하세요.\n- 반드시 모든 답변을 한글로 작성하세요.\n\n# Steps\n\n1. 입력 재료만 활용 가능한 음식 2가지를 선정하고, 서로 비슷하지 않은지 확인하세요.\n2. reasoning(논리/설명) 필드에: \n    - 해당 재료로 어떤 음식이 가능한지, 그 이유를 간단히 단계별 논리로 설명하세요.\n3. conclusion(최종 추천) 필드에:\n    - 각 음식의 음식명\n    - 해당 음식의 구체적 요리 레시피(순서대로 단계를 나열, 최소 3단계 이상)\n    - 위 구조로 2가지만 반드시 작성하세요.\n\n# Output Format\n\n모든 답변은 아래 JSON 구조로 출력하세요.  \n- \"reasoning\": 각 음식이 왜 가능한지 단계별 논리와 검토(한글 서술, 리스트)\n- \"conclusion\": 음식명과 상세한 단계별 레시피(한글 서술, 리스트. 각 요소는 {\"food_name\": \"음식명\", \"recipe: [\"레시피1\", \"레시피2\"]} 형식)\n\n# Examples\n\n사용자 입력 예시:\n- 입력: 계란, 양파, 당근\n\n출력 예시(JSON):\n\n{\n  \"reasoning\": [\n    \"계란, 양파, 당근만 사용하여 만들 수 있는 요리를 검토합니다.\",\n    \"계란과 채소(양파, 당근)만으로 달걀전이 가능합니다. 채소를 잘게 썰어 계란과 섞어 부치면 완성할 수 있습니다.\",\n    \"계란찜 역시 이 재료로 만들 수 있습니다. 계란을 풀고 다진 채소를 섞은 후, 찜기를 사용해 익히면 완성됩니다.\"\n  ],\n  \"conclusion\": [\n    {\n      \"food_name\": \"달걀전\",\n      \"recipe\": [\n        \"1. 양파와 당근을 잘게 썰어줍니다.\",\n        \"2. 계란을 풀고 썰어둔 양파와 당근, 소금, 후추를 넣고 섞습니다.\",\n        \"3. 달궈진 팬에 기름을 두르고 반죽을 얇게 올린 후, 앞뒤로 노릇하게 부칩니다.\"\n      ]\n    },\n    {\n      \"food_name\": \"계란찜\",\n      \"recipe\": [\n        \"1. 계란을 볼에 넣고 곱게 풀어줍니다.\",\n        \"2. 다진 양파와 당근, 소금, 후추를 계란물에 넣고 섞습니다.\",\n        \"3. 뚝배기나 내열 용기에 재료를 옮겨 담고, 중탕 또는 전자레인지로 익혀 부드럽게 완성합니다.\"\n      ]\n    }\n  ]\n}\n\n(실제 예시는 입력 재료와 음식에 따라 달라지며, 각 음식의 레시피 단계는 3단계 이상, 충분히 구체적으로 작성하십시오.)\n\n# Notes\n\n- 반드시 입력 재료만 사용하고, 음식명 및 조리법 전부 한글로 기입하세요.\n- 각 추천 요리는 서로 다른 종류여야 하며, 각 음식마다 레시피 단계는 구체적이고 논리적으로 작성돼야 합니다.\n- reasoning(논리/설명) → conclusion(최종 추천 및 레시피) 순서는 꼭 지켜야 합니다.\n- 답변 형식은 반드시 JSON이어야 하며, 한글로만 작성하세요.\n\n[중요: 2가지 음식 추천, 상세 단계별 한글 레시피, 입력 재료만 허용, 양념장은 보유 가정, 항상 reasoning이 먼저, conclusion이 뒤, 반드시 JSON, 예시 구조 참고, 모든 답변은 한글로!]"
                    }
                ]
            },
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'text',
                        'text': user_input
                    }
                ]
            }
        ],
        response_format={'type':'json_object'},
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0,
        store=False
    )

    return json.loads(response.choices[0].message.content)

In [29]:
chef_res = (chef_json("계란, 양파, 삼겹살"))
chef_res

{'reasoning': ['입력 재료는 계란, 양파, 삼겹살이므로 돼지고기와 양파를 볶는 요리와 계란을 부드럽게 익히는 요리를 만들 수 있습니다.',
  '삼겹살 양파볶음은 삼겹살에서 기름이 나오므로 별도의 식용유 없이 삼겹살과 양파를 볶고 기본양념으로 간을 할 수 있습니다.',
  '계란찜은 계란에 물과 다진 양파를 섞어 익히면 만들 수 있으며, 잘게 썬 삼겹살을 곁들여도 재료가 조화됩니다.',
  '두 음식은 하나는 고기를 볶는 요리이고 다른 하나는 계란을 찌는 요리이므로 조리법과 식감이 서로 다릅니다.'],
 'conclusion': [{'food_name': '삼겹살 양파볶음',
   'recipe': ['1. 삼겹살을 먹기 좋은 크기로 자르고, 양파는 껍질을 벗겨 굵게 채 썹니다.',
    '2. 달군 팬에 삼겹살을 올려 중간 불에서 앞뒤로 노릇하게 굽습니다. 삼겹살에서 기름이 나오므로 별도의 기름은 넣지 않아도 됩니다.',
    '3. 삼겹살이 충분히 익으면 양파를 넣고 함께 볶습니다.',
    '4. 양파가 투명해질 때까지 볶은 뒤 간장, 설탕, 후추를 넣고 골고루 섞습니다.',
    '5. 양파가 부드러워지고 양념이 삼겹살에 배면 불을 끄고 완성합니다.']},
  {'food_name': '삼겹살 양파 계란찜',
   'recipe': ['1. 삼겹살을 아주 작게 썰고, 양파도 잘게 다집니다.',
    '2. 팬에 삼겹살을 넣고 충분히 익을 때까지 볶은 뒤, 키친타월 등으로 지나치게 많은 기름만 덜어냅니다.',
    '3. 계란을 그릇에 깨 넣고 물과 소금, 후추를 넣어 잘 풉니다.',
    '4. 계란물에 다진 양파와 볶은 삼겹살을 넣고 고르게 섞습니다.',
    '5. 혼합물을 내열 용기나 뚝배기에 담고 약한 불에서 뚜껑을 덮어 천천히 익힙니다.',
    '6. 계란물이 가운데까지 굳고 표면이 촉촉하게 익으면 불을 끄고 잠시 뜸을 들인 후 냅니다.']}]}

In [30]:
output = chef_json("계란, 양파, 삼겹살, 찬법")

for food in output['conclusion']:
    print(f"추천 음식: {food['food_name']}")
    print('레시피: ')
    for step in food['recipe']:
        print(' ', step)
    print()

추천 음식: 양파 삼겹살볶음
레시피: 
  1. 삼겹살을 먹기 좋은 크기로 자르고, 양파는 껍질을 벗겨 굵게 썹니다.
  2. 달군 팬에 삼겹살을 올려 중간 불에서 굽고, 삼겹살에서 기름이 충분히 나오도록 뒤집어가며 익힙니다.
  3. 삼겹살이 노릇하게 익으면 양파를 넣고 삼겹살 기름에 함께 볶습니다.
  4. 양파가 투명해지고 부드러워지면 간장, 설탕, 후추를 넣고 재료에 양념이 고르게 배도록 볶습니다.
  5. 삼겹살이 완전히 익었는지 확인한 뒤 불을 끄고 그릇에 담습니다.

추천 음식: 양파 계란찜
레시피: 
  1. 양파를 잘게 다집니다.
  2. 계란을 그릇에 깨 넣고 소금과 후추를 넣어 충분히 풉니다.
  3. 다진 양파를 계란물에 넣고 고르게 섞습니다.
  4. 계란물을 내열 용기나 뚝배기에 담고, 찜기에서 약한 불로 천천히 익힙니다.
  5. 계란물이 거의 굳으면 젓가락으로 가운데를 찔러 익은 정도를 확인하고, 묽은 계란물이 나오지 않을 때 불을 끕니다.



In [45]:
def job_interview_json(user_input, /, *, model='gpt-5.6-luna', temperature=1, top_p=1, max_completion_tokens=2048):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': [
                    {
                        'type': 'text',
                        'text': """
    # Instruction
    당신은 20년 경력의 ML/DL 엔지니어이고, 이번 신입개발자 채용의 면접관이다.
    주어진 job posting(회사정보)를 바탕으로 신입개발자에 제공할 면접 질문과 모범 답변을 작성하세요.
    크게 hard skill과 soft skill/leadership 질문을 구분하여 제시하십시오.

    아래 지침을 반드시 준수하세요:

    - 면접 질문 및 답변은 반드시 한글로, 그리고 신입 개발자에게 현실적으로 맞도록 작성합니다.
    - hard skill(기술 질문)과 soft skill/leadership(소통, 태도, 리더십 등) 영역을 구분해 각각 최소 2개 이상의 예시를 만듭니다.
    - 입력 정보가 부족한 경우, 통상적인 산업/포지션 상황에 근거해 논리적으로 추론해 주세요.
    - 답변 전체의 output format은 반드시 아래 JSON 형태여야 하며, reasoning > hard_skill > soft_skill_leadership 순서로, 각 영역에 반드시 "질문"과 "모범답변"이 쌍으로 들어갑니다.
    - 입력값 미제공 시 [회사정보], [스펙] 등의 placeholder를 사용하세요.

    # Output Format

    아래 JSON 포맷으로 여는 코드블럭 없이 반드시 출력합니다:

    {
    "hard_skill": [
        {
        "question": "[hard skill(기술 중점) 면접 질문]",
        "answer": "[신입개발자 입장에서 모범답변]"
        },
        …
    ],
    "soft_skill_leadership": [
        {
        "question": "[soft skill/leadership(태도, 커뮤니케이션, 성장잠재력 등) 면접 질문]",
        "answer": "[신입개발자 입장에서 모범답변]"
        },
        …
    ]
    }

    - 각 영역별 면접 질문과 모범답변은 모두 한글로 작성합니다.
    - 반드시 하드 스킬, 소프트 스킬/리더십 항목을 구분하여 각 2개 이상 작성합니다(총 4개 이상).
    - 예시(Example)처럼 구조, 문장 길이, 구체성, 답변 스타일을 맞추되, 지원자는 신입개발자임을 꼭 반영하세요.

    # 예시(Example)

    Input:
    회사정보: [AI 기업, 제조공정 데이터 분석 솔루션 개발]
    스펙: [전자공학 전공, 신입 개발자, Python/Java 가능, 인턴 경험 있음]

    Output:
    {
    "hard_skill": [
        {
        "question": "Python 언어를 사용하여 데이터 전처리 경험이 있나요? 예시를 들어 설명해 주세요.",
        "answer": "대학 시절 프로젝트에서 Pandas를 이용해 결측치 처리와 이상치 제거를 해본 경험이 있습니다. 데이터 정제의 중요성을 실제로 체감할 수 있었습니다."
        },
        {
        "question": "제조 데이터와 같이 구조적인 데이터를 다룰 때 주의해야 하는 점은 무엇이라고 생각하나요?",
        "answer": "데이터의 정확한 구조 파악과, 이상치 및 에러를 사전에 점검하는 것이 중요하다고 생각합니다."
        }
    ],
    "soft_skill_leadership": [
        {
        "question": "팀 프로젝트에서 갈등이 있을 때 어떻게 해결해보셨나요?",
        "answer": "인턴 경험 중 팀원과 의견이 달랐을 때 서로의 입장을 경청한 뒤 중간 지점을 찾아 협력한 경험이 있습니다."
        },
        {
        "question": "빠르게 변화하는 환경에서 새로운 기술을 습득했던 경험이 있나요?",
        "answer": "새로운 도구가 필요했던 프로젝트에서 적극적으로 온라인 자료를 찾아 공부하며 습득하였습니다."
        }
    ]
    }

    # Notes
    - 지원자는 신입개발자, 면접관 페르소나는 20년차 ml/dl 엔지니어임을 모든 답변에 일관되게 반영하세요.
    - 모든 콘텍스트와 답변, 지시문, reasoning, 사례, placeholder, Q&A 등은 전체 한글로 작성합니다.
    - 포맷(최상단 하드스킬 > 소프트스킬/리더십)과 다국어 혼용 불허, 신입 시점 미반영 등은 모두 미승인 처리 대상입니다.

    **Objective Reminder:**
    1. '20년차 ml/dl engineer' 페르소나로서, 신입 개발자 대상 면접 Q&A를 회사/직무 정보로부터 reasoning → hard/soft skill/leaderhip별 제시
    2. JSON 아웃풋 포맷만 사용 (코드블럭 넣지 않음)
                        """
                    }
                ]
            },
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'text',
                        'text': user_input
                    }
                ]
            }
        ],
        response_format={'type':'json_object'},
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0,
        store=False
    )

    return json.loads(response.choices[0].message.content)

In [46]:
output = job_interview_json("""
## Job Descriptions

주요 업무 내용 안내

### 코딕스 개발팀 소개말

코딕스 개발팀은 콘텐츠 플랫폼 테스트북을 중심으로 일하고 있습니다. 사용자들이 더 편하게 서비스를 이용하도록 돕고있어요. 테스트북 뿐만 아니라 웹을 기반으로 한 다양한 서비스를 개발합니다.
기존에 사용하던 도구 이외에 새로운 Framework나 기술에도 관심이 많으며 적극적으로 검토하고 도입하기 위해 노력해요. 물론 개인의 역량과 커리어 증진에도 힘쓰고 있습니다.

---

## AI 개발자 주요 업무 내용 안내

### AI 개발자 (AI Developer)

AI 기술을 이용한 교육용 애플리케이션 프로젝트를 개발(ML/DL/Generative AI)하고,
Python 기반 API 서버 개발 및 Kubernetes 기반 배포 환경 구성합니다.

---

### 기술 및 자격 요건

* Python 프로그래밍 (FastAPI, Django, Flask 등) 경험이 있는 사람
* NLP 프로젝트 또는 LLM + RAG + VectorDB 기반 개발 경험이 있는 사람
* RESTful API 설계 및 Kubernetes 기반 배포 경험이 있는 사람
* 업무 커뮤니케이션에 대한 소통 능력이 뛰어난 분

---

### 우대 사항

* 바이브 코딩 경험을 보유하신 분
* 생성형 AI 관련 프로젝트 경험이 있는 사람
* 각종 협업 도구에 익숙하고 새로운 사용에 적극적인 사람
""")

In [47]:
output

{'reasoning': '코딕스 개발팀은 교육용 콘텐츠 플랫폼과 웹 기반 서비스를 개발하므로, 신입 개발자에게는 인공지능 모델 자체의 지식뿐 아니라 파이썬 기반 응용 프로그램 개발, 검색 증강 생성 구조, 벡터 데이터베이스, 응용 프로그램 인터페이스 설계, 쿠버네티스 배포에 대한 기초 이해가 중요합니다. 또한 교육 서비스는 사용자의 이해도와 편의성이 중요하므로, 기술적인 판단을 쉽게 설명하고 다른 직군과 협업하는 능력을 함께 평가해야 합니다. 신입 지원자에게는 실무 경험의 규모보다 문제를 구조화하고 학습하며 검증한 과정을 중심으로 질문하는 것이 적절합니다.',
 'hard_skill': [{'question': '교육용 생성형 인공지능 서비스를 개발한다고 할 때, 검색 증강 생성의 기본 흐름을 설명해 주세요.',
   'answer': '먼저 교육 콘텐츠를 문서 단위 또는 의미가 있는 문단 단위로 나누고, 각 문단을 임베딩하여 벡터 데이터베이스에 저장합니다. 사용자의 질문도 임베딩한 뒤 유사한 문서를 검색하고, 검색 결과와 질문을 언어 모델에 함께 전달하여 답변을 생성하는 방식으로 이해하고 있습니다. 실제 서비스에서는 검색 결과의 정확도, 출처 표시, 잘못된 답변 방지와 개인정보 보호도 함께 고려해야 합니다.'},
  {'question': '파이썬 기반 웹 프레임워크를 사용해 인공지능 기능을 응용 프로그램 인터페이스로 제공한다면, 어떤 점을 설계하겠습니까?',
   'answer': '먼저 요청과 응답 형식을 명확히 정의하고, 입력값 검증과 오류 응답 형식을 일관되게 설계하겠습니다. 예를 들어 질문을 받아 답변을 반환하는 경로를 만들고, 인증이 필요한 경우 사용자 확인 절차도 추가하겠습니다. 또한 언어 모델 호출은 시간이 오래 걸릴 수 있으므로 시간 제한, 예외 처리, 요청 기록을 적용하고, API 명세 문서를 작성해 프런트엔드 개발자와 쉽게 협업할 수 있도록 하겠습니다. 신입인 만큼 기존 팀의 설계 규칙과 코드 리뷰를 적극적으로 참고하겠습니다.'},
  

In [50]:
for category, qa_list in output.items():
    print(f'[{category}]')
    if category == 'reasoning':
        print(qa_list)
        print()

    else:
        for qa in qa_list:
            question = qa.get('question', '').strip()
            answer = qa.get('answer', '').strip()

            print(f'Q: {question}')
            print(f'A: {answer}')
        print()

[reasoning]
코딕스 개발팀은 교육용 콘텐츠 플랫폼과 웹 기반 서비스를 개발하므로, 신입 개발자에게는 인공지능 모델 자체의 지식뿐 아니라 파이썬 기반 응용 프로그램 개발, 검색 증강 생성 구조, 벡터 데이터베이스, 응용 프로그램 인터페이스 설계, 쿠버네티스 배포에 대한 기초 이해가 중요합니다. 또한 교육 서비스는 사용자의 이해도와 편의성이 중요하므로, 기술적인 판단을 쉽게 설명하고 다른 직군과 협업하는 능력을 함께 평가해야 합니다. 신입 지원자에게는 실무 경험의 규모보다 문제를 구조화하고 학습하며 검증한 과정을 중심으로 질문하는 것이 적절합니다.

[hard_skill]
Q: 교육용 생성형 인공지능 서비스를 개발한다고 할 때, 검색 증강 생성의 기본 흐름을 설명해 주세요.
A: 먼저 교육 콘텐츠를 문서 단위 또는 의미가 있는 문단 단위로 나누고, 각 문단을 임베딩하여 벡터 데이터베이스에 저장합니다. 사용자의 질문도 임베딩한 뒤 유사한 문서를 검색하고, 검색 결과와 질문을 언어 모델에 함께 전달하여 답변을 생성하는 방식으로 이해하고 있습니다. 실제 서비스에서는 검색 결과의 정확도, 출처 표시, 잘못된 답변 방지와 개인정보 보호도 함께 고려해야 합니다.
Q: 파이썬 기반 웹 프레임워크를 사용해 인공지능 기능을 응용 프로그램 인터페이스로 제공한다면, 어떤 점을 설계하겠습니까?
A: 먼저 요청과 응답 형식을 명확히 정의하고, 입력값 검증과 오류 응답 형식을 일관되게 설계하겠습니다. 예를 들어 질문을 받아 답변을 반환하는 경로를 만들고, 인증이 필요한 경우 사용자 확인 절차도 추가하겠습니다. 또한 언어 모델 호출은 시간이 오래 걸릴 수 있으므로 시간 제한, 예외 처리, 요청 기록을 적용하고, API 명세 문서를 작성해 프런트엔드 개발자와 쉽게 협업할 수 있도록 하겠습니다. 신입인 만큼 기존 팀의 설계 규칙과 코드 리뷰를 적극적으로 참고하겠습니다.
Q: 벡터 데이터베이스를 사용하는 검색 시스템의 결과가 좋지 않을 때, 어떤 순서로 원인을 확인하겠습니까?
A: 먼저